<a href="https://colab.research.google.com/github/ModularUS/dasIT/blob/main/beamform_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<h1>Plane Wave Beamforming Tutorial</h1>

This is a hands-on introduction to plane wave ultrasound beamforming. During this exercise we will look at the raw ultrasound data acquired and how to convert it into actual images. 

<br>

<h2>Main Take Aways</h2>

- understand ultrasonic raw data,
- know the necessary processing elements of a plane wave beamformer, and
- be able to perform a quality estimation of an ultrasound image.

<br>

<h2>Supplemental Information</h2>

This practical example is part of the ***Graz University of Technology*** lecture series ***Development of Electronic Systems*** and the ***Fundamentals of Biomedical Engineering Laboratory***. 

This program is free software and licensed under the ***Apache License v2.0*** (see the github LICENSE file for details). It is also citable: ***Leitner, C. (2022). New directions in recording and processing electro-mechanical signals from the human body. Dissertation. Graz University of Technology.***

<br>

<h2>Uncomplete List of Other Free Ultrasound Ressources:</h2>

*   <a href="http://www.k-wave.org/">k-wave ultrasound simulator</a> - free MATLAB and C++ implementations
*   <a href="https://field-ii.dk/">field II ultrasound simulator</a> - free MATLAB implementation
*   <a href="https://github.com/Sergio5714/pybf">pybf - Python beamformer</a> - optimized for short processing times
*   <a href="https://www.biomecardio.com/MUST/">MATLAB ultrasound toolbox</a> - free MATLAB beamformer<br><br>

<br><br>

I wish everyone a great dive into the topic, and please do not hesitate to <a href="mailto:christoph.leitner@tugraz.at">contact</a> me in case of **any** questions!

yours,<br>
Christoph<br><br>

<br>
<br>


---




# Introduction


<br>

**But what is beamforming anyway?** Essentially, this involves generating ultrasonic waves (beam patterns) for a transmit event (TX) and reconstructing images from the reflected ultrasonic waves or received signals (RX) on the piezo. 

<br>

**Simplifications:** For ultrasound image reconstruction, we make some additional assumptions, e.g..
- sound propagates throughout the tissue at a constant speed of sound,
- sound propagates in straight lines,
- sound is attenuated by the same amount throughout the tissue.

However, all of these assumptions are only model simplifications. Therefore, we must be aware that image distortion can occur if one of these basic assumptions are not met.

<br>

<h2>Plane wave beamforming principles</h2>

This method of beamforming is probably the easiest way to get started. In **plane wave beamforming**, all transducer elements are activated simultaneously, creating a single, unfocused ultrasonic plane wave that propagates in a straight line into the tissue (see figure below). 

<br>

<figure>
<center>
<img src='https://raw.githubusercontent.com/ModularUS/dasIT/main/example_data/ColabBFfigures/PlaneWaveBF.gif'/ width="1200">
</figure>

<br>

**How does this simplify things?** With a beamformer we have to solve the inverse problem, i.e. the transfer of the ultrasonic signal amplitudes from the time-channel space into an image space. In the case of beamforming plane waves, the solution simplifies to straightforward geometric formulations (we will have a look at them later on).

<br>

---

# 1.Getting Started


```
[#] Areas shown like this are executable code. Use the mousover play button to run these cells.
```

## Installation
First we need to install the beamformer package from the the <a href="https://github.com/ModularUS/dasIT">GitHub repository</a>:

In [ ]:
# Fetch the dasIT package from the github repository

!pip install git+https://github.com/ModularUS/dasIT

Next we load all other necessary libraries into our Colab notebook:

In [ ]:
# import necessary packages
import os
import time
import urllib.request
import numpy as np
from matplotlib import pyplot as plt

# to run dasIT, we need to import the necessary commands from the library
from dasIT.data.loader import load_rf
from dasIT.geometry import linear_array, plane_wave, plane_waves
from dasIT.signal import bandpass, analytic, oversample, envelope, log_compress
from dasIT.beamform import Beamformer
from dasIT.plot import show_image

## Download the example dataset

In [ ]:
# Download the dataset from github (skipped if the data is already there)

# The data was obtained in the course of the work for:
# Leitner et al. 2020, "Detection of Motor Endplates in Deep and Pennate Skeletal Muscles in-vivo using Ultrafast Ultrasound",
# 2020 IEEE International Ultrasonics Symposium (IUS).
#

rfdata_path = 'example_data/CIRSphantom_GE9LD_VVantage/CIRS_phantom.h5'

if os.path.exists(rfdata_path) == False:
  os.makedirs(os.path.dirname(rfdata_path))
  urllib.request.urlretrieve('https://raw.githubusercontent.com/ModularUS/dasIT/main/' + rfdata_path, rfdata_path)

### Compilation of the data set
Five frames were captured on a **Verasonics Vantage 256** ultrasound research system, using a **GE-9LD** transducer, and a **CIRS General Purpose** ultrasound phantom.

The figure below shows the reconstructed ultrasonic image (left) captured on an ultrasound phantom (center) with a GE-9LD transducer (right).

<figure>
<center>
<img src='https://raw.githubusercontent.com/ModularUS/dasIT/main/example_data/ColabBFfigures/ExperimentalSetup.jpg'/ width="1200">
</figure>

# 2.Define Hardware and Imaging Medium

To get started, we need to provide our software several static parameters. In particular, we have to define the used ultrasound hardware (e.g. transducer design,..), the emitted ultrasound signal (e.g. used ultrasound frequency,..) and the medium below the ultrasound lens (speed of sound, the pixels we want to compute).

We assume a constant speed of sound (**v = 1540 m/s**) in the tissue. This model assumption allows us to infer the tissue depth of the reflected echo by the relationship v=s/t and the measurement of the delay time of the reflected sound (Puls-Echo method).

All values are given in SI units (metres, seconds, Hertz).

## Transducer

Based on the transducer specs given above fill in the necessary parameters below:

In [ ]:
# Transducer and acquisition
center_frequency = 5.2083e6    # <--- CENTER FREQUENCY OF THE TRANSMIT PULSE IN [Hz]
bandwidth = (3.3125e6, 7.2875e6)   # lower and upper band edge of the transducer [Hz]
n_elements = 192               # <--- NUMBER OF TRANSDUCER ELEMENTS [#]
element_pitch = 2.3e-4         # <--- ELEMENT PITCH IN [m]
sampling_frequency = 4 * center_frequency   # the scanner takes four samples per period [Hz]

# x position of every element [m], x = 0 is the centre of the transducer
element_x = linear_array(n_elements, element_pitch)

print(f'Sampling frequency: {sampling_frequency / 1e6:.2f} MHz')
print(f'Transducer width:   {1e3 * (element_x[-1] - element_x[0]):.1f} mm, from {1e3 * element_x[0]:.2f} to {1e3 * element_x[-1]:.2f} mm')

### When was the first sample taken?

To convert a time into a depth we must know the time of the first sample, counted from the moment the wave leaves the transducer surface. Two settings of the scanner decide this. Both are given in wavelengths:

- The scanner starts to record when echoes from a depth of **5 wavelengths** come back. That is 2 x 5 periods after the transmit (the way down and the way back).
- The sound has to cross the **lens** of the transducer twice. The lens of the GE-9LD delays every echo by 2 x 3.38 periods.

The first sample therefore belongs to a time of 2 x (5 - 3.38) periods = 0.62 microseconds. This looks small, but it moves the whole image by half a millimetre.

In [ ]:
start_depth_wavelengths = 5.0                       # Receive.startDepth of the scanner
lens_wavelengths = 3.3820129870129874               # Trans.lensCorrection of the GE-9LD
time_zero = 2 * (start_depth_wavelengths - lens_wavelengths) / center_frequency   # time of sample 0 [s]

print(f'Time of the first sample: {1e6 * time_zero:.2f} microseconds')

## Medium

Next we define the speed of sound and the pixels of our image. The size of an image is often specified in "wavelength numbers" via the relationship lambda = c/f. We choose

- in depth: 177 wavelengths (about 52 mm), four pixels per wavelength,
- in width: the width of the transducer, three pixels per element.

In [ ]:
# Medium
speed_of_sound = 1540.0        # <--- SPEED OF SOUND IN [m/s]
wavelength = speed_of_sound / center_frequency

# Pixel grid [m]
grid_z = np.arange(177 * 4) * wavelength / 4
grid_x = np.linspace(element_x[0], element_x[-1], 3 * (n_elements - 1) + 1)

print(f'Wavelength: {1e3 * wavelength:.3f} mm')
print(f'Image: {grid_z.size} x {grid_x.size} pixels, {1e3 * grid_z[-1]:.1f} mm deep, {1e3 * (grid_x[-1] - grid_x[0]):.1f} mm wide')

# 3.Data Wrangling

From an ultrasound research system, we usually get more information than we actually need (due to the memory allocation of the system). The file holds five frames. Every frame is one plane wave transmit, recorded on all 192 elements: a table with one row per time sample and one column per element.

In [ ]:
### Load RF Data
RFdata = load_rf(rfdata_path)          # [samples, channels, shots, frames]
rf = RFdata[:, :, 0, 0]                # the first frame

last_sample = np.flatnonzero(np.abs(rf).sum(axis=1))[-1]

print(f'Samples per channel: {RFdata.shape[0]}')
print(f'Channels of transducer: {RFdata.shape[1]}')
print(f'Number of frames: {RFdata.shape[3]}')
print(f'Last sample that holds data: {last_sample} (the rest of the reserved memory is empty)')

## Exploring raw data - Beamforming Principle

If we now plot the sorted and trimmed channel data over time (left image), point-like echoes (phantom reflectors in the right image) show up, hyperbolic in the x,t plots.

**With our beamformer, we now need to solve the inverse problem and transfer the raw ultrasound data (time and channel space) into an image (width and height space)**.

<figure>
<center>
<img src='https://raw.githubusercontent.com/ModularUS/dasIT/main/example_data/ColabBFfigures/Beamformer_xt_xz.jpg'/ width="1200">
</figure>

### Exploring data of a single channel
Let's have a look at the raw data of **Channel 156**. We plot the signal amplitude over time.

The very bright band in the first samples is the **echo of the transducer surface and lens**. It is many times stronger than any echo from the phantom. Keep it in mind, we will meet it again in the image.

In [ ]:
channel = 156
dbrange = 55
shown = slice(0, last_sample + 1)

fig = plt.figure(figsize=(10, 4), dpi=100)
ax_1 = fig.add_subplot(121)
ax_1.imshow(log_compress(np.abs(rf[shown]), dbrange, reference=np.abs(rf[100:]).max()),
            aspect='auto',
            cmap='gray')

ax_1.set_xlabel('Transducer Element [#]', fontsize=15, fontweight='bold', labelpad=10)
ax_1.set_ylabel('Passing Time (Sample [#])', fontsize=15, fontweight='bold', labelpad=10)
ax_1.xaxis.tick_top()
ax_1.xaxis.set_label_position('top')
ax_1.axvline(x=channel, color='red')

ax_2 = fig.add_subplot(122)
ax_2.plot(rf[shown, channel], 'r')
ax_2.set_xlabel('Samples [#]')
ax_2.set_ylabel('Signal [ADC counts]')
ax_2.set_title(f'RF-data channel {channel}')

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

print('')
print('Channel 156 is indicated in red color.')

# 4.Signal Processing

### Time Gain Compensation

We assume that acoustic waves are absorbed to the same extent everywhere in the tissue (which is not formally correct, since different tissue types absorb to different amounts). The pressure amplitudes of acoustic waves lose energy as they propagate through tissue due to attenuation. Therefore, reflections from deeper structures appear weaker. For this reason, a time-dependent gain is applied that compensates for the attenuation losses.

<figure>
<center>
<img src='https://raw.githubusercontent.com/ModularUS/dasIT/main/example_data/ColabBFfigures/TGC.jpg'/ width="1200">
</figure>

The scanner already applied such a gain in its amplifiers while it recorded this data set. We therefore apply no further gain here.

###Filtering

To eliminate unwanted noise, we filter our raw data with a bandpass filter. We use the upper and lower limits of the transducer bandwidth as cutoff frequencies. The filter runs forward and backward over the signal, so it does not shift the echoes in time.

In [ ]:
### Filter RF Data
rf_filtered = bandpass(rf, bandwidth, sampling_frequency)

In [ ]:
# Plot channel 156
frequency = np.fft.rfftfreq(rf.shape[0], 1 / sampling_frequency) / 1e6
spectrum = lambda signal: 20 * np.log10(np.abs(np.fft.rfft(signal)) + 1e-12)

fig = plt.figure(figsize=(6, 5), dpi=100)
ax_1 = fig.add_subplot(311)
ax_1.plot(rf[shown, channel])
ax_1.set_xlabel('Samples [#]')
ax_1.set_ylabel('Signal')
ax_1.set_title(f'RF-data channel {channel}')

ax_2 = fig.add_subplot(312)
ax_2.plot(rf_filtered[shown, channel], 'r')
ax_2.set_xlabel('Samples [#]')
ax_2.set_ylabel('Signal')
ax_2.set_title(f'Filtered RF-data channel {channel}')

ax_3 = fig.add_subplot(313)
ax_3.plot(frequency, spectrum(rf[:, channel]), label='raw')
ax_3.plot(frequency, spectrum(rf_filtered[:, channel]), 'r', label='filtered')
ax_3.axvspan(bandwidth[0] / 1e6, bandwidth[1] / 1e6, color='gray', alpha=0.2)
ax_3.set_xlabel('Frequency [MHz]')
ax_3.set_ylabel('Amplitude [dB]')
ax_3.set_title(f'Spectrum channel {channel} (gray: transducer band)')
ax_3.legend(loc='upper right')

plt.tight_layout()
plt.show()

### Convert to analytical signal

In the ultrasound domain, we prefer to use signal envelopes to represent data We display the instantaneous energy distributions from fluctuating raw signals. Therefore, we must first calculate the analytical signal of the raw data using the Hilbert transformation.

In [ ]:
####################################################################
#------------------------ Analytical Signal -----------------------#

### Hilbert Transform
rf_analytic = analytic(rf_filtered)

In [ ]:
# Plot channel 156 around the echo of a wire
zoom = slice(450, 650)

fig = plt.figure(figsize=(5, 3), dpi=100)
ax_1 = fig.add_subplot(211)
ax_1.plot(np.arange(zoom.start, zoom.stop), rf[zoom, channel])
ax_1.set_xlabel('Samples [#]')
ax_1.set_ylabel('Signal')
ax_1.set_title(f'RF-data channel {channel}')

ax_2 = fig.add_subplot(212)
ax_2.plot(np.arange(zoom.start, zoom.stop), rf_analytic[zoom, channel].real, 'r')
ax_2.plot(np.arange(zoom.start, zoom.stop), envelope(rf_analytic[zoom, channel]), 'g', label='envelope')
ax_2.set_xlabel('Samples [#]')
ax_2.set_ylabel('Signal')
ax_2.set_title(f'Analytic signal channel {channel}')
ax_2.legend(loc='lower right')

plt.tight_layout()
plt.show()

### More samples per period

Our data has only four samples per period. The beamformer will need the signal at times that lie *between* two samples. Reading the nearest sample instead can be wrong by a large part of the amplitude. We therefore first compute a four times finer sampled version of the signal (16 samples per period). No information is added by this step. The signal between the samples is fully determined by the samples, as long as it contains no frequencies above half the sampling frequency.

In [ ]:
oversampling = 4
rf_fine = oversample(rf_analytic, oversampling)
fs_fine = oversampling * sampling_frequency

fig = plt.figure(figsize=(5, 2.5), dpi=100)
few = np.arange(520, 532)
plt.plot(np.arange(few[0] * oversampling, few[-1] * oversampling + 1) / oversampling, rf_fine[few[0] * oversampling:few[-1] * oversampling + 1, channel].real, 'r.-', label='16 samples per period')
plt.plot(few, rf_analytic[few, channel].real, 'ko', label='recorded samples')
plt.xlabel('Samples [#]')
plt.ylabel('Signal')
plt.legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()

# 5.Beamforming


To solve the beamforming operation, signal amplitudes must be transferred from the (x,time) domain into the (z,x) plane. The figure below depicts the necessary mathematical operations to calculate the time delays needed to determine
the signal amplitudes at each pixel point:

<figure>
<center>
<img src='https://raw.githubusercontent.com/ModularUS/dasIT/main/example_data/ColabBFfigures/PWbeamforming.jpg'/ width="800">
</figure>


### When does the echo of a point arrive?

Take one pixel at the position (x, z) and one transducer element at x_element. Two times add up:

- the plane wave travels straight down and reaches the pixel after **z / c**,
- the echo travels back to the element along the distance **sqrt(z² + (x - x_element)²)**.

```
arrival time = z / c  +  sqrt(z² + (x - x_element)²) / c
sample       = (arrival time - time of the first sample) * sampling frequency
```

Elements further away from the pixel hear the echo later. Over all elements the arrival times form a hyperbola. Let us draw this hyperbola for the position of a wire of the phantom on top of the raw data:

In [ ]:
# A wire of the phantom sits at about x = 14 mm, z = 20 mm
x_wire, z_wire = 14.0e-3, 19.97e-3

arrival_time = z_wire / speed_of_sound + np.sqrt(z_wire ** 2 + (x_wire - element_x) ** 2) / speed_of_sound
arrival_sample = (arrival_time - time_zero) * sampling_frequency

window = slice(450, 850)
fig = plt.figure(figsize=(10, 4), dpi=100)
for k, with_curve in enumerate([False, True]):
  ax = fig.add_subplot(1, 2, k + 1)
  ax.imshow(log_compress(envelope(rf_analytic[window]), 40), aspect='auto', cmap='gray', extent=[0, n_elements, window.stop, window.start])
  if with_curve:
    ax.plot(np.arange(n_elements) + 0.5, arrival_sample, 'r--', lw=1)
  ax.set_ylim(window.stop, window.start)
  ax.set_xlabel('Transducer Element [#]')
  ax.set_ylabel('Sample [#]')
  ax.set_title('calculated arrival of the echo (red)' if with_curve else 'recorded echoes')
plt.tight_layout()
plt.show()

The calculated curve lies on the recorded echo of the wire. **This is the whole idea of delay-and-sum beamforming:** for every pixel we collect the samples along its hyperbola and add them up. If there is a reflector at the pixel, all samples along the curve have the same sign and the sum gets large. If there is none, they cancel.

### Acount for element directivity and focus with an apodization table

Due to the directivity of the elements, the transducer elements cannot transmit and receive equally in all directions. Therefore, only a partial aperture of the elements contributes to the signal amplitude of a pixel. We use the elements within a width of **depth / f-number** around the pixel (f-number = 1.5), and weight them with a Hann window: the element directly above the pixel counts fully, elements at the edge of the aperture count little.

### Setup the beamformer

The `Beamformer` gets everything we defined so far. `tx=plane_wave(0)` tells it that we transmitted an unsteered plane wave.

In [ ]:
####################################################################
#-------------------------- Beamformer ----------------------------#

beamformer = Beamformer(element_x=element_x,
                        grid_x=grid_x,
                        grid_z=grid_z,
                        fs=fs_fine,
                        c=speed_of_sound,
                        tx=plane_wave(0),
                        t0=time_zero,
                        f_number=1.5,
                        window='hann',
                        interp='linear',
                        normalise=True)

For every element the beamformer knows two tables with one value per pixel: the **delay table** (at which sample does the echo of this pixel arrive?) and the **apodization table** (how much does this element count for this pixel?). Have a look at both for one element:

In [ ]:
element = 60
extent = [1e3 * grid_x[0], 1e3 * grid_x[-1], 1e3 * grid_z[-1], 1e3 * grid_z[0]]

fig = plt.figure(figsize=(10, 4), dpi=100)
ax_1 = fig.add_subplot(121)
im = ax_1.imshow(beamformer.delays(element) / oversampling, extent=extent, aspect='equal')
ax_1.set_title(f'Delay table of element {element} [samples]')
fig.colorbar(im, ax=ax_1)

ax_2 = fig.add_subplot(122)
im = ax_2.imshow(beamformer.weights(element), extent=extent, aspect='equal', cmap='gray')
ax_2.set_title(f'Apodization table of element {element} [weight]')
fig.colorbar(im, ax=ax_2)

for ax in (ax_1, ax_2):
  ax.plot(1e3 * element_x[element], 0, 'rv', clip_on=False)
  ax.set_xlabel('Width [mm]')
  ax.set_ylabel('Depth [mm]')
plt.tight_layout()
plt.show()

print('The red triangle marks the element. The cone in the right image is the area this element contributes to.')

###Beamforming

Now we apply the apodization and delay tables to the processed raw ultrasound data. In this step, we solve the inverse problem and map the time-channel data to an image space.

In [ ]:
####################################################################
#-------------------------- Beamforming ---------------------------#
start_das_timing = time.time()

image = beamformer(rf_fine)

print(f'Beamformed image: {image.shape[0]} x {image.shape[1]} pixels in {time.time() - start_das_timing:.1f} s')

# 6.Image Formation

The so-called dynamic range defines the logarithmic ratio between the highest (strongest echo near the lens) and the lowest signal amplitude (weak echo from deep targets). Assuming the beamformed data is mapped linearly onto a grayscale image, significant tissue features are lost because a few echoes with high amplitude overshadow the rest of the image space. Therefore, we balance the dynamic range by logarithmically adjusting the echo amplitudes.

Remember the bright echo of the transducer surface in the raw data? It is in the image as well, in the first millimetre. If we used it as the brightest point of the image, the phantom would stay dark. We therefore take the strongest echo **below 3 mm** as the reference (0 dB).

That's it. Now we can plot the results graphically:

In [ ]:
####################################################################
#------------------------ Image Formation --------------------------

# Envelope
image_envelope = envelope(image)

# Logarithmic compression
dbrange = 58
reference = image_envelope[grid_z > 3e-3].max()
image_db = log_compress(image_envelope, dbrange, reference=reference)

# Plot image
fig = plt.figure(figsize=(7, 7), dpi=100)
ax = show_image(image_db, grid_x, grid_z, ax=fig.add_subplot(111))
ax.axhline(95 * 1e3 * wavelength, color='yellow', lw=0.8, ls='--')
plt.show()

print('The acquisition was set up for an image depth of 95 wavelengths (yellow line).')
print('Below it the image is still valid, but fewer and fewer elements contribute to a pixel.')

# 7.Calculate Image Resolution

For the indicated point target in the near field (figure below) calculate the axial and lateral resolutions of your beamformed image.

<figure>
<center>
<img src='https://raw.githubusercontent.com/ModularUS/dasIT/main/example_data/ColabBFfigures/Resolution.jpg'/ width="300">
</figure>


Instructions:
- Use a ruler and measure the full width (start and end points) of your reflection target in the lateral and axial directions.
- One full width is equals approximately 6 standard deviations (± 3$\sigma$ = 99.73%).
- Divide the full width by 6 to get the approximate value of one sigma.
- Calculate the full width at half maximum (FWHM) using a Gaussian distribution: $FWHM = 2 \sqrt{2 ln2} * \sigma$
- You can do the calculation in <a href="https://www.wolframalpha.com/">https://www.wolframalpha.com/</a>

### Check your result

We can also measure the full width at half maximum directly in the data. We take the wire at 20 mm depth, cut one line through its brightest pixel in each direction and measure the width of the peak at half of its height.

In [ ]:
def full_width_half_maximum(profile, step):
  # width of the peak of a profile at half of its maximum, in the unit of step
  fine = np.interp(np.linspace(0, len(profile) - 1, 50 * len(profile)), np.arange(len(profile)), profile)
  peak = fine.argmax()
  above = fine >= fine.max() / 2
  left, right = peak, peak
  while left > 0 and above[left - 1]:
    left -= 1
  while right < len(fine) - 1 and above[right + 1]:
    right += 1
  return (right - left) * step / 50

# the area around the wire
ix = np.flatnonzero(np.abs(grid_x - x_wire) < 2.0e-3)
iz = np.flatnonzero(np.abs(grid_z - z_wire) < 1.0e-3)
patch = image_envelope[np.ix_(iz, ix)]
peak_z, peak_x = np.unravel_index(patch.argmax(), patch.shape)

lateral_mm = full_width_half_maximum(patch[peak_z, :], 1e3 * (grid_x[1] - grid_x[0]))
axial_mm = full_width_half_maximum(patch[:, peak_x], 1e3 * (grid_z[1] - grid_z[0]))

fig = plt.figure(figsize=(9, 3), dpi=100)
ax_1 = fig.add_subplot(131)
ax_1.imshow(patch, extent=[1e3 * grid_x[ix[0]], 1e3 * grid_x[ix[-1]], 1e3 * grid_z[iz[-1]], 1e3 * grid_z[iz[0]]], cmap='gray', aspect='equal')
ax_1.set_xlabel('Width [mm]')
ax_1.set_ylabel('Depth [mm]')
ax_1.set_title('Wire')
ax_2 = fig.add_subplot(132)
ax_2.plot(1e3 * grid_x[ix], patch[peak_z, :] / patch.max())
ax_2.axhline(0.5, color='gray', ls='--')
ax_2.set_xlabel('Width [mm]')
ax_2.set_title(f'lateral: {lateral_mm:.2f} mm')
ax_3 = fig.add_subplot(133)
ax_3.plot(1e3 * grid_z[iz], patch[:, peak_x] / patch.max())
ax_3.axhline(0.5, color='gray', ls='--')
ax_3.set_xlabel('Depth [mm]')
ax_3.set_title(f'axial: {axial_mm:.2f} mm')
plt.tight_layout()
plt.show()

print(f'Wavelength: {1e3 * wavelength:.2f} mm')
print(f'Lateral resolution (FWHM): {lateral_mm:.2f} mm')
print(f'Axial resolution (FWHM):   {axial_mm:.2f} mm')

**Why are the two numbers so different?** The axial resolution is set by the length of the transmitted pulse (a few periods). The lateral resolution is set by the width of the aperture: roughly wavelength x f-number, and somewhat more with a Hann window.

Try it: change the `f_number` or the `window` (`'box'`) of the beamformer above, run the cells again and watch how the width of the wire changes.

# 8.Outlook: Compounding of Several Plane Waves

A single plane wave gives an image with one transmit, which is very fast. The price is contrast: without a transmit focus, echoes from other places leak into a pixel. A common remedy is to send several plane waves, each **steered by a small angle**, and to add the beamformed images (coherent compounding).

For a steered wave only the transmit part of the delay changes:

```
transmit time = ( z cos(angle) + x sin(angle) ) / c
```

Our data set holds one angle only. We therefore simulate the echoes of two point reflectors for seven angles and compare the image of the unsteered wave with the compounded image.

In [ ]:
# Simulated transducer and reflectors
sim_fs, sim_fc, sim_c = 40e6, 5e6, 1540.0
sim_elements = linear_array(128, 0.3e-3)
points = [(0.0, 15e-3), (4e-3, 22e-3)]         # (x, z) [m]
angles = [-9, -6, -3, 0, 3, 6, 9]              # [degrees]
waves = plane_waves(angles)

def simulate(wave, n_samples=1600):
  # echoes of the points for one transmitted wave: a short pulse at the arrival time on every element
  t = np.arange(n_samples)[:, None] / sim_fs
  record = np.zeros((n_samples, sim_elements.size))
  for x_point, z_point in points:
    arrival = wave(x_point, z_point, sim_c) + np.sqrt(z_point ** 2 + (x_point - sim_elements) ** 2) / sim_c
    record += np.exp(-((t - arrival) * sim_fc / 1.5) ** 2) * np.cos(2 * np.pi * sim_fc * (t - arrival))
  return record

shots = analytic(np.stack([simulate(wave) for wave in waves], axis=2))      # [samples, channels, shots]

# Beamform: one wave, and all three together
sim_x = np.arange(-10e-3, 10e-3, 0.1e-3)
sim_z = np.arange(10e-3, 27e-3, 0.05e-3)
settings = dict(element_x=sim_elements, grid_x=sim_x, grid_z=sim_z, fs=sim_fs, c=sim_c, f_number=1.5, window='box')
unsteered = angles.index(0)
single = envelope(Beamformer(tx=waves[unsteered], **settings)(shots[:, :, unsteered]))
compound = envelope(Beamformer(tx=waves, **settings)(shots))

fig = plt.figure(figsize=(12, 4), dpi=100)
for k, (result, title) in enumerate([(single, 'one plane wave (0 degrees)'), (compound, f'{len(angles)} plane waves compounded')]):
  show_image(log_compress(result, 50), sim_x, sim_z, ax=fig.add_subplot(1, 3, k + 1), title=title)
ax = fig.add_subplot(133)
row = np.argmin(np.abs(sim_z - points[0][1]))
ax.plot(1e3 * sim_x, 20 * np.log10(single[row] / single[row].max() + 1e-6), label='one plane wave')
ax.plot(1e3 * sim_x, 20 * np.log10(compound[row] / compound[row].max() + 1e-6), label='compounded')
ax.set_ylim(-60, 0)
ax.set_xlabel('Width [mm]')
ax.set_ylabel('Amplitude [dB]')
ax.set_title('Line through the upper reflector')
ax.legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()

Look at the line through the upper reflector: next to the main peak the compounded image is darker. Echoes that leak in from the side arrive with a different delay for every angle and cancel in the sum.

Try it: use fewer angles, e.g. `angles = [-9, 0, 9]`, or a wider range. How many angles are needed before the result stops improving?